In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.set(style='darkgrid', font_scale=1)

In [ ]:
train_df = pd.read_csv('/kaggle/input/tabular-playground-series-sep-2022/train.csv')
test_df = pd.read_csv('/kaggle/input/tabular-playground-series-sep-2022/test.csv')
sample_df = pd.read_csv('/kaggle/input/tabular-playground-series-sep-2022/sample_submission.csv')

In [ ]:
train_df.head()

In [ ]:
# Checking for null values
train_df.isnull().sum()

In [ ]:
#Formatting date column
train_df['date'] = pd.to_datetime(train_df['date'], format='%Y-%m-%d')
test_df['date'] = pd.to_datetime(test_df['date'], format='%Y-%m-%d')

### EDA
* You can find other notebooks focusing more on EDA and we can learn a lot from them on our time series feature correlation, seaonality , trend
* https://www.kaggle.com/code/cabaxiom/tps-sep-22-eda-and-linear-regression-baseline a great notebook for EDA
* https://www.kaggle.com/code/samuelcortinhas/tps-sept-22-timeseries-analysis#4.-Seasonality this notebook was really helpfull for FE

In [ ]:
def val_count_df(df, column_name, sort_by_column_name=False):
    value_count = df[column_name].value_counts().reset_index().rename(columns={column_name:"Value Count","index":column_name}).set_index(column_name)
    value_count["Percentage"] = df[column_name].value_counts(normalize=True)*100
    value_count = value_count.reset_index()
    if sort_by_column_name:
        value_count = value_count.sort_values(column_name)
    return value_count



def plot_and_display_valuecounts(df, column_name, sort_by_column_name=False):
    val_count = val_count_df(df, column_name, sort_by_column_name)
    #display(val_count)
    val_count.set_index(column_name).plot.pie(y="Value Count", figsize=(5,5), legend=False, ylabel="");

In [ ]:
plot_and_display_valuecounts(train_df, "product")

In [ ]:
plot_and_display_valuecounts(train_df, "store")

In [ ]:
plot_and_display_valuecounts(train_df, "country")

In [ ]:
plt.figure(figsize=(12,5))
ax = sns.lineplot(data=pd.DataFrame(train_df.groupby(['country','date']).sum()['num_sold']/train_df.groupby(['date']).sum()['num_sold']), x='date', y='num_sold', hue='country')
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.title('Ratio of sales by country')
plt.ylabel('Ratio')
plt.show()

* Beginning in 2020, the ratios converge to same value, so we should add a 'pandamic year' feature for our data 

In [ ]:
plt.figure(figsize=(12,5))
ax = sns.lineplot(data=train_df.groupby(['product','date']).sum()/train_df.groupby(['date']).sum(), x='date', y='num_sold', hue='product')
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.title('Ratio of sales by product')
plt.xticks(rotation=70)
plt.ylabel('Ratio')
plt.show()

* The proportion of books sold by product has a strong seasonal pattern with a time period of 2 years (look at red curve).
* More importantly, this pattern doesn't change during the year 2020.
* We can extrapolate this trend to the year 2021 using Fourier features.

In [ ]:
product_df = train_df.groupby(["date","product"])["num_sold"].sum().reset_index()
f,ax = plt.subplots(figsize=(20,10))
sns.lineplot(data=product_df, x="date", y="num_sold", hue="product");

* Motivation behind adding a 'is_special' and 'is_weekend' features where we see a large increase in sales independently on product, sales or country

# Preprocessing date column

* Merging in order to process train and test data simultaneously

In [ ]:
ntrain = train_df.shape[0]
ntest = test_df.shape[0]
y_train = train_df['num_sold'].values
test_IDs = test_df['row_id'].copy()
all_data = pd.concat((train_df, test_df)).reset_index(drop=True)
all_data.drop(['row_id','num_sold'], axis=1, inplace=True)
print("all_data size is : {}".format(all_data.shape))

In [ ]:
all_data.head()

In [ ]:
important_dates = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12,16,17, 124, 125, 126, 127, 140, 141,142, 167, 
                   168, 169, 170, 171, 173, 174, 175, 176, 177, 178, 179, 180, 181, 203, 230, 231, 232,
                   233, 234, 282, 289, 290, 307, 308, 309, 310, 311, 312, 313, 317, 318, 319, 320, 360,
                   361, 362, 363, 364, 365]


all_data['year'] = all_data['date'].dt.year
all_data['month'] = all_data['date'].dt.month.astype('category')
all_data['week'] = all_data['date'].dt.isocalendar().week.astype('int')
all_data['day'] = all_data['date'].dt.day.astype('category')

all_data['day_of_year'] = all_data['date'].dt.day_of_year
all_data["day_of_year"] = all_data.apply(lambda x: x["day_of_year"]-1 if (x["date"] > pd.Timestamp("2020-02-29") and x["date"] < pd.Timestamp("2021-01-01"))  else x["day_of_year"], axis=1)

all_data["important_dates"] = all_data["day_of_year"].apply(lambda x: x if x in important_dates else 0).astype('category')

all_data["day_of_week"] = all_data["date"].dt.dayofweek
all_data["day_of_week"] = all_data["day_of_week"].apply(lambda x: 0 if x<=3 else(1 if x==4 else (2 if x==5 else (3))))

all_data["is_weekend"] = all_data["date"].dt.day_of_week.apply(lambda x: 1 if x in [5,6] else (0)).astype('category')
all_data['is_special'] = (all_data['date'].dt.month*100+all_data['date'].dt.day).isin([101,102,1228,1229,1230,1231]).astype('int').astype('category')

# We observed changes in year 2020 so it would be helpful to add it as a feature

#all_data['quarantine'] = ((all_data['year'].astype(int) == 2020) & (all_data['date'].dt.month.astype(int) >= 0) & (all_data['date'].dt.month.astype(int) <= 4)).astype('category')

all_data['is_pandemic_year'] = all_data['year'].astype(int) >= 2020

#Fourier features allow use sine and cosine waves to model seasonality patterns in the data.
#We can change the time period to model yearly, monthly and weekly seasonal patterns. 
#The motivation for this is that people spend money differently depending on whether it is 
#the summer vs winter, or weekday vs weekend.

all_data["month_cos"] = np.cos(all_data['date'].dt.month * (2 * np.pi / 12))
all_data["month_sin"] = np.sin(all_data['date'].dt.month * (2 * np.pi / 12))

all_data.drop(['date'] , axis = 1 , inplace = True)

In [ ]:
all_data.head()

# Label encoding our categorical columns

In [ ]:
from sklearn_pandas import DataFrameMapper
from sklearn.preprocessing import StandardScaler , LabelBinarizer , Normalizer

In [ ]:
categorical_cols = list(all_data.select_dtypes(exclude='number').columns)
categorical_cols

In [ ]:
for col in categorical_cols:
    all_data[col] = all_data[col].astype("category").cat.codes.astype("category")
all_data[categorical_cols].head()

# Splitting train and test

In [ ]:
train_data = all_data[:ntrain].copy()
test_data = all_data[ntrain:]

train_data['num_sold'] = y_train

train_data.shape , test_data.shape

# Appliying log to normalize target column
* Normalization is a part of data processing and cleansing techniques. The main goal of normalization is to make the data homogenous over all records and fields. It helps in creating a linkage between the entry data which in turn helps in cleaning and improving data quality.
* Let's try two methods : first manually by applying log(x + a) while increasing a till our data is as normalized as possible
* The Second one is is using scipy's boxcox module, which proves to perform better than the first method in removing the skewness of our label data

In [ ]:
# Just for clarity when using log(x+a) formula for normalization
x = train_data['num_sold']

from scipy.stats import norm, skew #for some statistics
print(f"Skewness of our target data before normalization : {skew(x)}")

In [ ]:
from scipy.stats import probplot, boxcox
from scipy.special import inv_boxcox
 
# transform training data & save lambda value
fitted_data, fitted_lambda = boxcox(x)
  
# plotting the original data(non-normal) and
# fitted data (normal)
sns.displot(x, kde = True,
            label = "Non-Normal", color ="green")
 
sns.displot(fitted_data, kde = True,
            label = "Normal", color ="green")
 
# adding legends to the subplots
plt.legend(loc = "upper right")
 
print(f"Lambda value used for Transformation: {fitted_lambda}")

In [ ]:
skew(fitted_lambda)

In [ ]:
train_data['num_sold'] = fitted_data

## Checking Disribution of continuous features

In [ ]:
continuous_cols = list(train_data.select_dtypes(include='number').columns)
continuous_cols.remove('num_sold')
continuous_cols

In [ ]:
feat_float = continuous_cols
fig, axes = plt.subplots(2, 4, figsize = (20,10))
for i, ax in enumerate(axes.reshape(-1)):
    if i < len(feat_float):
        sns.kdeplot(x = feat_float[i], data = train_data, fill = True, ax = ax)
        ax.tick_params()
        ax.xaxis.get_label().set_fontsize(20)
        ax.set_ylabel('')
fig.suptitle('Distribution of features', color="#3a0ca3",fontsize = 35, x = 0.5, y = 1)
plt.tight_layout()
plt.show()

# Model selection

In [ ]:
X_train = train_data.drop('num_sold', axis = 1)
y_train = train_data.num_sold
X_train.shape , y_train.shape

In [ ]:
X_train.head()

In [ ]:
import xgboost as xg
import lightgbm as lgb
from sklearn.pipeline import Pipeline
from catboost import CatBoostRegressor
from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV , cross_val_score ,cross_validate
from sklearn.preprocessing import StandardScaler

In [ ]:
# Adding Smape scoring for cross_val
from sklearn.metrics import make_scorer

def smape(a, f):
    return 1/len(a) * np.sum(2 * np.abs(f-a) / (np.abs(a) + np.abs(f))*100)

scoring = {'neg_mean_absolute_error': 'neg_mean_absolute_error',
           'smape': make_scorer(smape, greater_is_better= False),
          }

In [ ]:
my_smape = make_scorer(smape, greater_is_better= False)

In [ ]:
pipeline_cat = Pipeline([('CatBoostRegressor', CatBoostRegressor(random_state=42 ,
                                                                 verbose = 0, 
                                                                 cat_features= [0,1,2,9,10,11]))])

pipeline_xgbr = Pipeline([ ('XGBRegressor', xg.XGBRegressor(random_state=42,
                                                            enable_categorical = True,
                                                            tree_method =  "hist"))])

pipeline_lgbm = Pipeline([('LGBMRegressor', lgb.LGBMRegressor(verbose = -1))])
pipeline_lgbm_scaled = Pipeline([('scaler',StandardScaler()),
                                 ('LGBMRegressor', lgb.LGBMRegressor(verbose = -1))])

pipeline_lasso = Pipeline([('Lasso', Lasso())])
pipeline_lasso_scaled = Pipeline([('scaler',StandardScaler()),
                                  ('Lasso', Lasso())])


pipelines = [ 
    pipeline_lgbm,
    pipeline_xgbr,
    pipeline_lasso,
    pipeline_lasso_scaled,
    pipeline_lgbm_scaled,
    #pipeline_cat
            ]

pipe_dict = {
    0: 'LGBMRegressor',
    1: 'XGBRegressor',
    2: 'Lasso',
    3: 'Lasso_Scaled',
    4: 'LGBM_scaled'
    #2: 'CatBoostRegressor', 
            }

In [ ]:
import time

original_results = dict()
for i, model in enumerate(pipelines):
    start = time.time()
    # cross_validate to use custom scoring, however can't use .mean() since it returns a dict
    cv_score = cross_val_score(model, X_train,y_train, cv=5,scoring= my_smape ).mean()
    original_results[pipe_dict[i]] = cv_score
    
    end = time.time()
    print(f" model {pipe_dict[i]} took : {end - start} s ")

In [ ]:
for key , value in sorted(original_results.items(),key=lambda item : item[1]):
    print(key, value)

> #### => LGBM is the faster and best performing model

# Applying Grid search for Hyper-parameters Tuning

* Note : LightGBM can handle categorical features automatically,

In [ ]:
results = dict()

#GradientBoostingRegressor
lgbm = lgb.LGBMRegressor(objective ='regression',
                         metric = 'mae',
                         random_state = 42)

#USING GRID SEARCH
params_lgbm = {
    # The maximum number of leaves per tree; higher num_leaves means less conservative/control, potentially overfitting (default is 31)
    'num_leaves':[23,],
    # Lower means longer to train but more accurate # Should not interfere with overfitting (default is 0.1)
    'learning_rate':[0.02024883654139076,],
    # The more trees the less likely the algorithm is to overfit. So try increasing the number of estimators. (default is 100)
    'n_estimators':[7042,],
    # The ratio of features used (i.e. columns used); colsample_bytree. Lower ratios avoid over-fitting. (default is 1.0)
    'colsample_bytree': [0.5416121298384964,],
    # The ratio of the training instances used (i.e. rows used); subsample. Lower ratios avoid over-fitting. (default is 1.0)
    'subsample': [0.7524541459326278,],
    #Penalize too small weights by L1 and high weights (== outliers) by L2 to prevent overfitting. 
    'reg_alpha': [1.9643990765525134e-09,], # L1 regularization (default is 0.0)
    'reg_lambda': [7.544556675033748e-07], # L2 regularization (default is 0.0)
    # This controls the complexity of branching; Decreasing this value prevents overfitting. (default is -1)
    'max_depth': [13],
               } 

grid_search_lgbm = GridSearchCV(estimator=lgbm, 
                                param_grid=params_lgbm,
                                verbose = 1, 
                                cv = 5,
                                scoring= my_smape, 
                                n_jobs=-1).fit(X_train, y_train.values.ravel())

lgbm_best = grid_search_lgbm.best_estimator_
print('lgbm Regressor Best Parmas',grid_search_lgbm.best_params_)
print('lgbm Regressor Best Score',grid_search_lgbm.best_score_)

In [ ]:
def plotImp(model, X , num = 20, fig_size = (40, 20)):
    feature_imp = pd.DataFrame({'Value':list(model.feature_importances_),
                                'Feature':list(X.columns)})
    plt.figure(figsize=fig_size)
    sns.set(font_scale = 5)
    sns.barplot(x="Value", y="Feature", data=feature_imp.sort_values(by="Value", 
                                                        ascending=False)[0:num])
    plt.title('LightGBM Features (avg over folds)')
    plt.tight_layout()
    plt.show()
    
plotImp(lgbm_best , X_train)

In [ ]:
pipeline = Pipeline([('estimator', lgbm_best)])
cv_results = cross_validate(pipeline, X_train, y_train, cv=5, scoring=scoring, return_train_score=True, return_estimator=True)
print(np.mean(cv_results['test_neg_mean_absolute_error']))
print(np.mean(cv_results['test_smape']))

In [ ]:
scores = np.zeros(test_df.shape[0])
for estimator in cv_results['estimator']:
    scores += inv_boxcox(estimator.predict(test_data), fitted_lambda)
    
scores /= len(cv_results['estimator'])

In [ ]:
sub = pd.DataFrame()
sub['row_id'] = test_IDs
sub['num_sold'] = scores
sub.to_csv('submission.csv',index=False)
sub